In [8]:
from IPython.display import HTML
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import numpy as np
import pickle

from draw import *

with open('LipsArrayHalfTime.np', 'rb') as f:
    LipsData = pickle.load(f)

def lagrange_interpolate(t_old, y_old, t_new):
    n = len(t_old)
    y_new = np.zeros(len(t_new))
    for i in range(n):
        L = np.ones(len(t_new))
        for j in range(n):
            if i != j:
                L *= (t_new - t_old[j]) / (t_old[i] - t_old[j])
        y_new += y_old[i] * L
    return y_new

In [9]:
def cubic_spline_interpolate(t_old, y_old, t_new):
    n = len(t_old)
    h = np.diff(t_old)
    A = np.zeros((n, n))
    B = np.zeros(n)
    A[0, 0] = 1.0
    A[n-1, n-1] = 1.0
    
    for i in range(1, n-1):
        A[i, i-1] = h[i-1]
        A[i, i] = 2 * (h[i-1] + h[i])
        A[i, i+1] = h[i]
        B[i] = 3 * ((y_old[i+1] - y_old[i]) / h[i] - (y_old[i] - y_old[i-1]) / h[i-1])

    c = np.linalg.solve(A, B)
    a = y_old[:-1]
    b = np.zeros(n-1)
    d = np.zeros(n-1)

    for i in range(n-1):
        b[i] = (y_old[i+1] - y_old[i]) / h[i] - h[i] * (c[i+1] + 2*c[i]) / 3
        d[i] = (c[i+1] - c[i]) / (3 * h[i])
        
    y_new = np.zeros(len(t_new))
    for k, t in enumerate(t_new):
        if t == t_old[-1]:
            i = n - 2
        else:
            i = np.searchsorted(t_old, t, side='right') - 1
            i = max(0, min(i, n - 2))
        dt = t - t_old[i]
        y_new[k] = a[i] + b[i]*dt + c[i]*dt**2 + d[i]*dt**3
        
    return y_new

SplineLipsLetters = {}
for letter, frames in LipsData.items():
    n = frames.shape[0]
    t_old = np.arange(n)
    t_new = np.arange(0, n - 0.5, 0.5) 
    frames_flat = frames.reshape(n, 80)
    new_frames_flat = np.zeros((len(t_new), 80))
    
    for coord_idx in range(80):
        y_old = frames_flat[:, coord_idx]
        new_frames_flat[:, coord_idx] = cubic_spline_interpolate(t_old, y_old, t_new)
        
    SplineLipsLetters[letter] = new_frames_flat.reshape(len(t_new), 40, 2)

animate_lips(SplineLipsLetters['А'], interval=100)

In [10]:
word = "АБЁБА"
parts = []

for letter in word:
    parts.append(LipsData[letter][1:-1]) 

word_frames = np.concatenate(parts, axis=0)
n_word = word_frames.shape[0]
t_old_word = np.arange(n_word)
t_new_word = np.arange(0, n_word - 0.5, 0.5)
n_new_word = len(t_new_word)
word_frames_flat = word_frames.reshape(n_word, 80)
word_lagrange_flat = np.zeros((n_new_word, 80))
word_spline_flat = np.zeros((n_new_word, 80))

for coord_idx in range(80):
    y_old = word_frames_flat[:, coord_idx]
    word_lagrange_flat[:, coord_idx] = lagrange_interpolate(t_old_word, y_old, t_new_word)
    word_spline_flat[:, coord_idx] = cubic_spline_interpolate(t_old_word, y_old, t_new_word)
    
WordLagrange = word_lagrange_flat.reshape(n_new_word, 40, 2)
WordSpline = word_spline_flat.reshape(n_new_word, 40, 2)

In [11]:
from IPython.display import display

print("Метод Лагранжа:")
display(animate_lips(WordLagrange, interval=100))

print("Метод сплайнов:")
display(animate_lips(WordSpline, interval=100))

Метод Лагранжа:


Метод сплайнов:
